In [ ]:
from google.colab import drive

drive.mount("/content/drive")


# Coleta de pareceres de PEC na Camara

Notebook Colab para clonar o repositorio, instalar dependencias e executar o coletor `coleta.camara.pareceres_pec.collect` salvando dados no Google Drive. O modulo coleta documentos de parecer de PEC na CCJC/CCJR historica, em comissoes especiais de PEC e no Plenario, incluindo voto em separado e parecer vencedor quando a tramitacao explicita esses casos.


In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)

for name in ["raw", "checkpoints", "logs", "manifests", "processed"]:
    (DATA_ROOT / name).mkdir(parents=True, exist_ok=True)

print("FALANDO_NELA_DATA_ROOT=", os.environ["FALANDO_NELA_DATA_ROOT"])


In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_DIR = Path("/content/falando_nela")
REPO_REF = ""  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

subprocess.run(["git", "-C", str(REPO_DIR), "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Repo em:", Path.cwd())
subprocess.run(["git", "status", "--short"], check=True)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


## Parametros

Use a validacao curta primeiro. A janela de marco de 2015 cobre um caso conhecido em que um parecer original passou a constituir voto em separado depois da aprovacao de parecer vencedor.


In [ ]:
from datetime import datetime, timezone

DATA_INICIO_VALIDACAO = "2015-03-01"
DATA_FIM_VALIDACAO = "2015-03-31"
SAMPLE_LIMIT_VALIDACAO = 12

DATA_INICIO_PROD = "2011-05-18"
DATA_FIM_PROD = "2026-05-18"

RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ID_VALIDACAO = f"colab-camara-pareceres-pec-validacao-{RUN_STAMP}"
RUN_ID_PROD = "prod-camara-pareceres-pec-baseline"

print("RUN_ID_VALIDACAO=", RUN_ID_VALIDACAO)
print("RUN_ID_PROD=", RUN_ID_PROD, "# fixo para permitir retomada com --resume")


In [ ]:
MODULE = "coleta.camara.pareceres_pec.collect"

import json
import subprocess
import sys
from pathlib import Path


def run_collector(run_id, data_inicio, data_fim, *, sample=False, sample_limit=None, resume=True, stream=True):
    cmd = [
        sys.executable,
        "-m",
        MODULE,
        "--mode",
        "prod",
        "--run-id",
        run_id,
        "--data-inicio",
        data_inicio,
        "--data-fim",
        data_fim,
    ]
    if resume:
        cmd.append("--resume")
    if sample:
        cmd.append("--sample")
    if sample_limit is not None:
        cmd.extend(["--sample-limit", str(sample_limit)])

    print("Rodando:", " ".join(cmd))
    output_lines = []
    if stream:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="", flush=True)
            output_lines.append(line.rstrip("\n"))
        returncode = process.wait()
    else:
        result = subprocess.run(cmd, check=False, text=True, capture_output=True)
        returncode = result.returncode
        if result.stdout:
            print(result.stdout)
            output_lines.extend(result.stdout.splitlines())
        if result.stderr:
            print(result.stderr)
            output_lines.extend(result.stderr.splitlines())

    manifest_path = manifest_from_output(output_lines, run_id)
    if returncode != 0:
        print(f"Coletor terminou com returncode={returncode}; veja logs/autosave para diagnostico.")
    return manifest_path


def manifest_from_output(output_lines, run_id):
    for line in reversed(output_lines):
        candidate = line.strip()
        if candidate.endswith(".json"):
            path = Path(candidate)
            if path.exists():
                return path

    manifest_path = DATA_ROOT / "manifests" / f"{run_id}.json"
    if manifest_path.exists():
        return manifest_path

    autosave_path = DATA_ROOT / "manifests" / f"{run_id}.autosave.json"
    if autosave_path.exists():
        return autosave_path

    raise FileNotFoundError(f"Manifesto nao encontrado para run_id={run_id}")


def show_manifest(path):
    path = Path(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    keys = [
        "run_id",
        "source",
        "dataset",
        "status",
        "errors",
        "data_inicio",
        "data_fim",
        "record_counts",
        "partition_counts",
        "processed_records_loaded",
    ]
    print(json.dumps({key: data.get(key) for key in keys}, ensure_ascii=False, indent=2, sort_keys=True))
    return data


def tail_log_for_run(run_id, n=30):
    path = DATA_ROOT / "logs" / f"{run_id}.jsonl"
    print("Log:", path)
    if not path.exists():
        print("Ainda nao ha log para este run_id.")
        return
    for line in path.read_text(encoding="utf-8").splitlines()[-n:]:
        print(line)


## Validacao curta

Esta celula roda em `--mode prod` com janela mensal e `--sample-limit`. Ela nao usa `--sample`, porque a API da Camara pode exigir percorrer mais de uma pagina de proposicoes para chegar aos casos de parecer naquele mes.


In [ ]:
manifest_validacao_path = run_collector(
    RUN_ID_VALIDACAO,
    DATA_INICIO_VALIDACAO,
    DATA_FIM_VALIDACAO,
    sample=False,
    sample_limit=SAMPLE_LIMIT_VALIDACAO,
    resume=True,
)
manifest_validacao = show_manifest(manifest_validacao_path)


In [ ]:
from datetime import date


def inspect_jsonl(path, max_rows=8):
    path = Path(path)
    print("\nArquivo:", path)
    if not path.exists():
        print("Nao encontrado")
        return

    lines = path.read_text(encoding="utf-8").splitlines()
    print("linhas=", len(lines), "bytes=", path.stat().st_size)
    for line in lines[:max_rows]:
        record = json.loads(line)
        payload = record.get("payload", {})
        colegiado = payload.get("colegiado") if isinstance(payload, dict) else {}
        texto = payload.get("texto") or ""
        print({
            "record_type": record.get("record_type"),
            "partition": record.get("partition"),
            "source_id": record.get("source_id"),
            "IdProposicao": payload.get("IdProposicao"),
            "pec": f"{payload.get('SiglaTipo')} {payload.get('Numero')}/{payload.get('Ano')}",
            "documento_classe": payload.get("documento_classe"),
            "status_deliberativo": payload.get("status_deliberativo"),
            "vencido": payload.get("vencido"),
            "ambito": colegiado.get("ambito") if isinstance(colegiado, dict) else None,
            "sigla_orgao": colegiado.get("sigla") if isinstance(colegiado, dict) else None,
            "metodo_obtencao": payload.get("metodo_obtencao"),
            "texto_status": payload.get("texto_status"),
            "texto_chars": len(texto),
        })


validation_start = date.fromisoformat(DATA_INICIO_VALIDACAO)
metadata_path = DATA_ROOT / "raw" / "camara" / "pareceres_pec" / "metadata" / f"{RUN_ID_VALIDACAO}.jsonl"
corpus_path = (
    DATA_ROOT
    / "raw"
    / "camara"
    / "pareceres_pec"
    / f"ano={validation_start.year:04d}"
    / f"mes={validation_start.month:02d}"
    / f"{RUN_ID_VALIDACAO}.jsonl"
)

inspect_jsonl(metadata_path, max_rows=3)
inspect_jsonl(corpus_path, max_rows=8)


## Coleta completa

Esta celula roda o baseline inteiro em modo producao, com `run_id` fixo e `--resume`. Se o Colab cair ou uma particao falhar, rode a mesma celula novamente com o mesmo `RUN_ID_COMPLETO`.


In [ ]:
EXECUTAR_COLETA_COMPLETA = False
RUN_ID_COMPLETO = RUN_ID_PROD

if EXECUTAR_COLETA_COMPLETA:
    print("Rodando coleta completa com run_id fixo:", RUN_ID_COMPLETO)
    manifest_completo_path = run_collector(
        RUN_ID_COMPLETO,
        DATA_INICIO_PROD,
        DATA_FIM_PROD,
        sample=False,
        sample_limit=None,
        resume=True,
        stream=True,
    )
    manifest_completo = show_manifest(manifest_completo_path)
else:
    print("Altere EXECUTAR_COLETA_COMPLETA para True quando quiser iniciar a coleta completa.")


## Inspecao e retomada

Use esta celula para consultar manifesto, autosave e ultimas linhas de log do `run_id` fixo.


In [ ]:
autosave_completo_path = DATA_ROOT / "manifests" / f"{RUN_ID_COMPLETO}.autosave.json"
manifest_completo_path = DATA_ROOT / "manifests" / f"{RUN_ID_COMPLETO}.json"

if manifest_completo_path.exists():
    show_manifest(manifest_completo_path)
elif autosave_completo_path.exists():
    show_manifest(autosave_completo_path)
else:
    print("Ainda nao ha manifest/autosave para", RUN_ID_COMPLETO)

tail_log_for_run(RUN_ID_COMPLETO, n=30)
